201303开始 www.szse.cn
201804开始 docs.static.szse.cn

In [1]:
import akshare as ak

In [ ]:
ak.stock_sse_summary()

In [ ]:
df1 = ak.stock_szse_sector_summary(symbol="当年", date="202501").drop('项目名称-英文',axis=1)
df2 = ak.stock_szse_sector_summary(symbol="当年", date="202401").drop('项目名称-英文',axis=1)

In [ ]:
((((df1['成交金额-人民币元']/df1['交易天数']))/100000000)-(((df2['成交金额-人民币元']/df2['交易天数']))/100000000)).round(2)

In [ ]:
df1

In [ ]:
(((df1['成交金额-人民币元']/df1['交易天数']))/100000000).round(2)

In [4]:
ak.stock_szse_sector_summary(symbol="当月", date="201801").drop('项目名称-英文',axis=1)

,项目名称,交易天数,成交金额-人民币元,成交金额-占总计,成交股数-股数,成交股数-占总计,成交笔数-笔,成交笔数-占总计
0,合计,22,5697300301711,100.00,419037870973,100.00,283874045,100.00
1,农林牧渔,22,78174684013,1.37,7603401053,1.81,3710924,1.31
2,采矿业,22,68641846925,1.20,7099818485,1.69,3823942,1.35
3,制造业,22,3703918004155,65.01,262520499531,62.65,184983737,65.16
4,水电煤气,22,65248924954,1.15,6915439046,1.65,4339277,1.53
5,建筑业,22,113367282372,1.99,9928495211,2.37,6667427,2.35
6,批发零售,22,122124663130,2.14,10181863312,2.43,6944441,2.45
7,运输仓储,22,49028406035,0.86,4263965681,1.02,3267445,1.15
8,住宿餐饮,22,2343946739,0.04,358313892,0.09,198005,0.07
9,信息技术,22,631007736688,11.08,41072243408,9.80,30750440,10.83


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import BytesIO, StringIO

In [ ]:
def stock_szse_sector_summary(
    symbol: str = "当月", date: str = "202501"
) -> pd.DataFrame:
    """
    深圳证券交易所-统计资料-股票行业成交数据
    https://docs.static.szse.cn/www/market/periodical/month/W020220511355248518608.html
    :param symbol: choice of {"当月", "当年"}
    :type symbol: str
    :param date: 交易年月
    :type date: str
    :return: 股票行业成交数据
    :rtype: pandas.DataFrame
    """
    url = "https://www.szse.cn/market/periodical/month/index.html"
    r = requests.get(url)
    r.encoding = "utf8"
    soup = BeautifulSoup(r.text, features="lxml")
    tags_list = soup.find_all(name="div", attrs={"class": "g-container"})[1].find_all(
        "script"
    )
    tags_dict = [
        eval(
            item.string[item.string.find("{") : item.string.find("}") + 1]
            .replace("\n", "")
            .replace(" ", "")
            .replace("value", "'value'")
            .replace("text", "'text'")
        )
        for item in tags_list
    ]
    date_url_dict = dict(
        zip(
            [item["text"] for item in tags_dict],
            [item["value"][2:] for item in tags_dict],
        )
    )
    date_format = "-".join([date[:4], date[4:]])
    url = f"https://www.szse.cn/market/periodical/month/{date_url_dict[date_format]}"
    r = requests.get(url)
    r.encoding = "utf8"
    soup = BeautifulSoup(r.text, features="lxml")
    url = [
        item for item in soup.find_all("a") if item.get_text() == "股票行业成交数据"
    ][0]["href"]

    if 'http' in url :
        pass
    else:
        url = 'https://www.szse.cn'+url

    if symbol == "当月":
        r = requests.get(url)
        r.encoding = 'GBK'
        temp_df = pd.read_html(StringIO(r.text), encoding="gbk")[0]
        temp_df.columns = [
            "项目名称",
            "项目名称-英文",
            "交易天数",
            "成交金额-人民币元",
            "成交金额-占总计",
            "成交股数-股数",
            "成交股数-占总计",
            "成交笔数-笔",
            "成交笔数-占总计",
        ]
    else:
        temp_df = pd.read_html(url, encoding="gbk")[1]
        temp_df.columns = [
            "项目名称",
            "项目名称-英文",
            "交易天数",
            "成交金额-人民币元",
            "成交金额-占总计",
            "成交股数-股数",
            "成交股数-占总计",
            "成交笔数-笔",
            "成交笔数-占总计",
        ]

    temp_df["交易天数"] = pd.to_numeric(temp_df["交易天数"], errors="coerce")
    temp_df["成交金额-人民币元"] = pd.to_numeric(
        temp_df["成交金额-人民币元"], errors="coerce"
    )
    temp_df["成交金额-占总计"] = pd.to_numeric(
        temp_df["成交金额-占总计"], errors="coerce"
    )
    temp_df["成交股数-股数"] = pd.to_numeric(temp_df["成交股数-股数"], errors="coerce")
    temp_df["成交股数-占总计"] = pd.to_numeric(
        temp_df["成交股数-占总计"], errors="coerce"
    )
    temp_df["成交笔数-笔"] = pd.to_numeric(temp_df["成交笔数-笔"], errors="coerce")
    temp_df["成交笔数-占总计"] = pd.to_numeric(
        temp_df["成交笔数-占总计"], errors="coerce"
    )
    return temp_df

In [ ]:
stock_szse_sector_summary(symbol="当月", date="201501")